# Exploratory Data Analysis — News Category Dataset

**Sprint 1 (S1-T7):** dataset shape, dtypes, missing values, duplicates, basic stats, class-distribution bar chart with imbalance commentary.

**Sprint 2 (S2-T9):** numeric distributions, boxplots, correlation heatmap, token-length distribution, top-20 most frequent words, n-gram analysis, word cloud, outlier visualisation, skewness check.

Run `00_main.ipynb` first to generate `data/processed/cleaned.parquet` (cheap on a warm runtime since the cleaned file is cached).

## Setup

In [ ]:
"""Sprint-1 EDA setup. Loads both the raw and the cleaned dataset
because some checks (missing values, duplicates, original column dtypes)
make sense only on the raw frame, while the per-class breakdowns work on
either.

Run `00_main.ipynb` first so `data/processed/cleaned.parquet` exists.
"""

import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "News_Category_Dataset_v3.json"
CLEANED_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned.parquet"
assert RAW_PATH.exists(), f"Missing {RAW_PATH} — run 00_main.ipynb data-cell first."
assert CLEANED_PATH.exists(), f"Missing {CLEANED_PATH} — run 00_main.ipynb preprocess-cell first."

raw_df = pd.read_json(RAW_PATH, lines=True)
clean_df = pd.read_parquet(CLEANED_PATH)
print(f"raw_df:   {raw_df.shape}")
print(f"clean_df: {clean_df.shape}")

## 1. Dataset Shape, Dtypes, Missing Values, Duplicates

In [ ]:
print(f"Shape: {raw_df.shape}\n")

print("Dtypes:")
print(raw_df.dtypes.to_string())

print("\nMissing values per column (raw):")
missing = raw_df.isnull().sum()
empty_str = (raw_df.select_dtypes(include="object").map(lambda v: isinstance(v, str) and v.strip() == "")).sum()
combined_missing = missing.add(empty_str, fill_value=0).astype(int).rename("missing_or_empty")
print(combined_missing.to_string())

print(f"\nDuplicated rows (raw): {raw_df.duplicated().sum():,}")
print(f"Duplicated headlines:  {raw_df['headline'].duplicated().sum():,}")

print("\nNumeric stats (cleaned):")
print(clean_df[["word_count", "char_count", "punct_count"]].describe().T.round(2))

**Findings (sprint 1 EDA, part 1).**

- **Size:** 209,527 articles across 6 raw columns. The dtypes are clean — five object (string) columns and one `datetime64[ns]` for `date`. No surprises.
- **Missing values:** concentrated in `authors` (17.9% missing) and `short_description` (9.4% missing). `headline` is essentially complete (6 missing), and the other columns are full. Practical impact: when we build the feature input we already concatenate `headline + " " + short_description` with `fillna("")`, so missing short descriptions degrade gracefully into headline-only rows.
- **Duplicates:** 13 full-row duplicates (negligible) and 1,531 duplicate headlines (~0.7%). The duplicate headlines are mostly identical breaking-news pieces re-published into more than one category; we leave them in for sprint 1 because removing them risks distorting the class balance. We will revisit if any model overfits.
- **Cleaned text length:** average 29 words / 174 characters per article after preprocessing, with a long tail (max 245 words). Comfortable for both TF-IDF (uni+bi-gram) and a 512-token BERT-family encoder.

## 2. Class Distribution + Imbalance Commentary

In [ ]:
counts = clean_df["category"].value_counts()
print(f"Number of distinct categories: {len(counts)}")
print(f"Largest class:  {counts.idxmax()} ({counts.max():,} articles)")
print(f"Smallest class: {counts.idxmin()} ({counts.min():,} articles)")
print(f"Imbalance ratio (max / min): {counts.max() / counts.min():.1f}x")
print(f"Median class size: {int(counts.median()):,}")

fig, ax = plt.subplots(figsize=(10, 11))
counts.sort_values().plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("Article count")
ax.set_ylabel("Category")
ax.set_title(f"News category distribution (n = {len(clean_df):,})")
plt.tight_layout()
plt.show()

**Class-imbalance commentary.**

- **42 categories** as the spec describes. The largest class — **POLITICS** — holds 35,602 articles (≈17% of the dataset) and the smallest — **EDUCATION** — only 1,014. The max-to-min ratio is **35.1×**, with a median class size of 3,362.
- The top three categories (POLITICS, WELLNESS, ENTERTAINMENT) together cover roughly **35%** of the dataset. The bottom 10 (everything from ARTS down) each have under 1,500 examples.
- **Modeling implication:** with 42 classes and severe imbalance, plain accuracy will look optimistic because rare classes are easily ignored by a classifier that always predicts the majority. Per **ADR-002** we use `class_weight="balanced"` for every model that supports it, and we report **macro-F1** prominently alongside weighted-F1 in the comparison table — macro-F1 weighs every class equally regardless of size, so it surfaces minority-class performance.
- **Honest expectation:** macro-F1 in the 0.55–0.65 range is realistic for classical TF-IDF on 42 classes with this imbalance (PRD §2 success target = 0.55 macro / 0.65 weighted). We will not overclaim if minority classes underperform; the written report will name them explicitly.

## Sprint 2 — Full Visualisation Set (S2-T9)

Numeric distributions, boxplots, correlation heatmap, token-length distribution, top-20 word frequencies, top bi-grams + tri-grams, word cloud, outlier visualisation, skewness + scaling-need checks.